# GUANACO general index-linked views

A linked workspace has one rule: a source plot emits stable IDs, and each target consumes those same IDs. The plots do not need application-specific glue. Links are directed from overview to detail.

## Data contract

| Input | Stable IDs | Use |
|---|---|---|
| `AnnData` | `obs_names` and `var_names` | cells and features |
| `MuData` | each modality's `obs_names` and `var_names` | shared cells or features |
| `DataFrame` | unique, non-null index | one atomic record per row |
| mapping | names to any objects above | multiple sources in one workspace |

An external table should be tidy/long: one row is the smallest record a user may select. An overview may aggregate many rows; GUANACO remembers which original indices belong to that mark.

## Minimal API

```python
gc.pl.link("source", "detail")                       # AnnData cells, or table rows
gc.pl.link("source", "detail", action="filter")    # keep only selected cells
gc.pl.link("table", "adata", by="cell")             # table index == obs_names
gc.pl.link("table", "adata", by="feature")          # table index == var_names
gc.pl.link("overview", "detail", key="pair_id")     # one logical ID → many rows
gc.pl.link("table", "adata", by="cell", key="cell_id")
```

Cell links highlight selected versus other cells by default. `action="filter"` removes unselected cells from the detail. Feature links replace the target feature context. Table-to-table links filter by the shared row index unless `key` names a repeated logical ID. For a mixed table–AnnData link, `key` maps table rows to `obs_names` or `var_names`.

After interacting in Jupyter or marimo, retrieve the IDs with `demo.get_selection("source_id")`; the returned object has `.by` and `.ids`.

In [ ]:
from pathlib import Path
import sys

import guanaco as gc

demo_candidates = [
    Path.cwd() / "examples" / "linked_views",
    Path.cwd().parent / "linked_views",
    Path.cwd() / "linked_views",
]
DEMO_DIR = next(path.resolve() for path in demo_candidates if (path / "demo_data.py").is_file())
if str(DEMO_DIR) not in sys.path:
    sys.path.insert(0, str(DEMO_DIR))

from demo_data import (
    load_liana_long,
    load_pbmc_cd4_relationship,
    load_pbmc_mudata_demo,
    load_spatial_relationship_demo,
    make_external_cell_table,
    make_pathway_demo,
    make_single_cell,
    make_volcano_data,
    significant_liana,
)

## Demo 1 · AnnData cells: UMAP → violin highlight

Lasso cells in the UMAP. Because both views use the same AnnData, the omitted `by` resolves to cells and the violin compares selected cells with all others.

In [ ]:
adata_1 = make_single_cell(seed=101)

demo_1 = gc.pl.linked_view(
    adata_1,
    title="Demo 1 · Cell selection → expression comparison",
    views=[
        gc.pl.umap(id="cells", color="cell_type", title="Cell embedding", height="420px"),
        gc.pl.violin(
            id="expression", keys=["IL7R"], groupby="cell_type",
            title="IL7R expression", height="420px",
        ),
    ],
    links=[gc.pl.link("cells", "expression")],
)
_ = demo_1.show_jupyter(port=8161, height=520)

## Demo 2 · AnnData cells: UMAP → filtered dotplot and heatmap

The explicit filter action recomputes both summaries from only the lassoed cells. Dot size is the actual expressing-cell fraction in that subset.

In [ ]:
adata_2 = make_single_cell(seed=202)
markers_2 = ["CD3D", "IL7R", "CCL5", "MS4A1", "NKG7", "LST1"]

demo_2 = gc.pl.linked_view(
    adata_2,
    title="Demo 2 · Selected cells → recomputed summaries",
    layout="grid",
    views=[
        gc.pl.umap(id="cells", color="cell_type", title="Cell embedding", height="360px"),
        gc.pl.dotplot(
            id="dotplot", var_names=markers_2, groupby="cell_type",
            title="Expression fraction and mean", height="360px",
        ),
        gc.pl.heatmap(
            id="heatmap", var_names=markers_2, groupby="cell_type",
            title="Cell-level expression", height="360px",
        ),
    ],
    links=[
        gc.pl.link("cells", "dotplot", action="filter"),
        gc.pl.link("cells", "heatmap", action="filter"),
    ],
)
_ = demo_2.show_jupyter(port=8162, height=720)

## Demo 3 · AnnData features: grouped matrix → UMAP

Click a feature tile in the grouped expression overview. A feature is not a cell selection, so `by="feature"` changes the feature displayed on the UMAP.

In [ ]:
adata_3 = make_single_cell(seed=303)
markers_3 = ["CD4", "IL7R", "CCL5", "MS4A1", "NKG7", "LST1", "CLEC10A"]

demo_3 = gc.pl.linked_view(
    adata_3,
    title="Demo 3 · Grouped feature overview → single-cell detail",
    views=[
        gc.pl.matrixplot(
            id="feature_summary", var_names=markers_3, groupby="cell_type",
            title="Grouped mean expression · click a tile", height="420px",
        ),
        gc.pl.umap(
            id="feature_umap", color="IL7R", title="Feature expression on UMAP",
            height="420px",
        ),
    ],
    links=[gc.pl.link("feature_summary", "feature_umap", by="feature")],
)
_ = demo_3.show_jupyter(port=8163, height=520)

## Demo 4 · MuData shared cells: RNA → protein

The two modalities share cell IDs but use genuinely different coordinates: RNA UMAP and protein PCA. Lasso RNA cells to highlight the same IDs in protein space. The helper uses the real PBMC MuData when available and a deterministic fallback otherwise.

In [ ]:
mdata_4, rna_feature_4, protein_feature_4, pbmc_path_4 = load_pbmc_mudata_demo()

demo_4 = gc.pl.linked_view(
    mdata_4,
    title="Demo 4 · RNA cells → matching protein cells",
    views=[
        gc.pl.umap(
            id="rna", data="rna", color=rna_feature_4,
            title=f"RNA · {rna_feature_4}", height="420px",
        ),
        gc.pl.pca(
            id="protein", data="protein", color=protein_feature_4,
            title=f"Protein PCA · {protein_feature_4}", height="420px",
        ),
    ],
    links=[gc.pl.link("rna", "protein")],
)
_ = demo_4.show_jupyter(port=8164, height=520)

## Demo 5 · External cell table → AnnData

The external table index equals `adata.obs_names`. Lasso independent per-cell attributes to highlight the matching cells on the UMAP.

In [ ]:
adata_5 = make_single_cell(seed=505)
cell_attributes_5 = make_external_cell_table(adata_5)

demo_5 = gc.pl.linked_view(
    {"attributes": cell_attributes_5, "cells": adata_5},
    title="Demo 5 · External cell attributes → expression space",
    views=[
        gc.pl.view(
            "plotly.scatter", id="attributes", data="attributes",
            x="library_complexity", y="activation_score", color="cell_type",
            title="Independent cell attributes", height="420px",
        ),
        gc.pl.umap(
            id="cell_umap", data="cells", color="cell_type",
            title="Matching AnnData cells", height="420px",
        ),
    ],
    links=[gc.pl.link("attributes", "cell_umap", by="cell")],
)
_ = demo_5.show_jupyter(port=8165, height=520)

## Demo 6 · External features: volcano → violin and heatmap

The volcano DataFrame index equals `adata.var_names`. Click a gene to update two GUANACO plot types without naming their internal feature parameters.

In [ ]:
adata_6, differential_6 = make_volcano_data()

demo_6 = gc.pl.linked_view(
    {"differential": differential_6, "cells": adata_6},
    title="Demo 6 · Differential feature → two expression details",
    layout="grid",
    views=[
        gc.pl.view(
            "plotly.scatter", id="volcano", data="differential",
            x="log2_fold_change", y="negative_log10_padj",
            color="result", label="feature", size=10,
            title="Stimulated vs Control · click a feature", height="360px",
        ),
        gc.pl.violin(
            id="feature_violin", data="cells", keys=["IL7R"],
            groupby="cell_type", title="Single-feature distribution",
            height="360px",
        ),
        gc.pl.heatmap(
            id="feature_heatmap", data="cells", var_names=["IL7R"],
            groupby="cell_type", title="Cell-level expression", height="360px",
        ),
    ],
    links=[
        gc.pl.link("volcano", "feature_violin", by="feature"),
        gc.pl.link("volcano", "feature_heatmap", by="feature"),
    ],
)
_ = demo_6.show_jupyter(port=8166, height=720)

## Demo 7 · PBMC LIANA atomic rows: network → interaction pairs

One LIANA interaction is one indexed row. The explicit p-value and rank cutoffs are applied before plotting; an empty result raises instead of silently substituting non-significant pairs. The network aggregates the retained rows by sender and receiver, and clicking an arrow filters the pair plot.

In [ ]:
liana_all_7, liana_path_7 = load_liana_long()
pairs_7 = significant_liana(liana_all_7, pvalue=0.05, rank=0.02)

demo_7 = gc.pl.linked_view(
    pairs_7,
    title="Demo 7 · Significant PBMC communication → interaction pairs",
    views=[
        gc.pl.view(
            "network", id="network", source="source", target="target",
            title="Communication overview · click an arrow", height="440px",
        ),
        gc.pl.view(
            "plotly.scatter", id="pairs", x="cell_pair", y="ligand_receptor",
            color="magnitude_rank", size="confidence", size_range=(9, 22),
            color_map="Viridis", title=f"High-confidence pairs · {len(pairs_7):,} rows",
            height="440px",
        ),
    ],
    links=[gc.pl.link("network", "pairs")],
)
print(f"Retained {len(pairs_7):,} of {len(liana_all_7):,} LIANA rows from {liana_path_7 or 'fallback data'}")
_ = demo_7.show_jupyter(port=8167, height=540)

## Demo 8 · Spatial relationship: enrichment → locations + co-occurrence

The overview is the complete `adata.uns['cluster_nhood_enrichment']['zscore']` matrix. A selected pair is a logical `pair_id`; it filters both the tissue spots and the matching `adata.uns['cluster_co_occurrence']` distance curve.

In [ ]:
pairs_8, spatial_8, cooccurrence_8, spatial_path_8 = load_spatial_relationship_demo()
z_limit_8 = max(1.0, float(pairs_8['enrichment'].abs().max()))

demo_8 = gc.pl.linked_view(
    {"pairs": pairs_8, "spatial": spatial_8, "cooccurrence": cooccurrence_8},
    title="Demo 8 · Neighborhood relationship → locations + co-occurrence",
    layout="grid",
    views=[
        gc.pl.view(
            "plotly.heatmap", id="neighborhoods", data="pairs",
            x="target_group", y="source_group", value="enrichment",
            color_map="RdBu_r",
            colorbar_title="Z-score", x_title="Neighbor group",
            y_title="Conditional group",
            title="Neighborhood enrichment", height="500px",
        ),
        gc.pl.spatial(
            id="locations", data="spatial", color="cluster",
            title="Source and target locations on tissue", size=3,
            height="500px",
        ),
        gc.pl.view(
            "plotly.line", id="cooccurrence", data="cooccurrence",
            x="distance", y="co_occurrence", group="pair_label",
            color="pair_label", color_mode="categorical",
            fixed_axes=True,
            title="Co-occurrence across distance · click a pair",
            height="500px",
        ),
    ],
    links=[
        gc.pl.link("neighborhoods", "locations", by="cell", key="cell_id", action="filter"),
        gc.pl.link("neighborhoods", "cooccurrence", key="pair_id"),
    ],
)
_ = demo_8.show_jupyter(port=8168, height=600)

In [ ]:
spatial_8

## Demo 9 · Real PBMC cross-omics relationship

RNA CD4 and protein CD4 form a second per-cell coordinate system. Lasso cells on the RNA UMAP to highlight their matching points. The legend identifies level-1 cell types with their Spearman ρ; the single shared colorbar uses one global −1…1 scale, so T-cell and monocyte correlations are directly comparable.

In [ ]:
(
    embedding_9, relationship_9, correlation_summary_9, pbmc_path_9,
    rna_feature_9, protein_feature_9,
) = load_pbmc_cd4_relationship(pbmc_path_4, mdata=mdata_4)

demo_9 = gc.pl.linked_view(
    {"embedding": embedding_9, "relationship": relationship_9},
    title="Demo 9 · RNA embedding → RNA/protein relationship",
    views=[
        gc.pl.umap(
            id="rna_umap", data="embedding", color="level1",
            title="PBMC RNA UMAP · level 1 cell type", height="440px",
        ),
        gc.pl.view(
            "plotly.scatter", id="cd4_relationship", data="relationship",
            x="rna_cd4", y="protein_cd4", color="level1_spearman",
            group="correlation_label", color_mode="continuous",
            color_map="RdBu_r", color_range=(-1, 1), color_midpoint=0,
            colorbar_title="Within-cell-type Spearman ρ",
            title="One shared cell per point · legend shows cell type and ρ",
            height="440px",
        ),
    ],
    links=[gc.pl.link("rna_umap", "cd4_relationship", by="cell")],
)
print(f"Data: {pbmc_path_9 or 'fallback data'}")
print(f"Features: RNA {rna_feature_9}; protein {protein_feature_9}")
print(correlation_summary_9[["level1", "n_cells", "spearman"]].to_string(index=False))
_ = demo_9.show_jupyter(port=8169, height=540)

## Demo 10 · Pathway overview → genes

Each horizontal pathway bar represents several genes. Clicking a bar sends all matching genes to the detail heatmap through `key="pathway"`. The detail view shows mean expression by cell type.


In [ ]:
pathways_10, genes_10 = make_pathway_demo()

demo_10 = gc.pl.linked_view(
    {"pathways": pathways_10, "genes": genes_10},
    title="Demo 10 · Pathway overview → genes",
    views=[
        gc.pl.view(
            "plotly.bar", id="pathways", data="pathways",
            x="pathway_score", y="pathway", orientation="h",
            color="pathway",
            title="Pathway overview · click a pathway", height="360px",
        ),
        gc.pl.view(
            "plotly.heatmap", id="genes", data="genes",
            x="gene", y="cell_type", value="gene_mean",
            color_map="Blues", colorbar_title="Mean expression",
            title="Genes in the selected pathway", height="460px",
        ),
    ],
    links=[gc.pl.link("pathways", "genes", key="pathway")],
)
_ = demo_10.show_jupyter(port=8170, height=560)


## Retrieve a notebook selection

After lassoing the source of Demo 9, run:

```python
selection = demo_9.get_selection("rna_umap")
selected_cell_ids = () if selection is None else selection.ids
```

The same call works for table rows and features; `.by` tells you which index axis was returned.